In [1]:
import pandas as pd
import numpy as np

In [2]:
data = {
    "Day": ["D1", "D2", "D3", "D4", "D5"],
    "Outlook": ["Sunny", "Sunny", "Sunny", "Sunny", "Sunny"],
    "Temperature": ["Mild", "Mild", "Mild", "Cool", "Mild"],
    "Humidity": ["Normal", "Normal", "Normal", "High", "High"],
    "Wind": ["Weak", "Weak", "Weak", "Weak", "Weak"],
    "Play": ["Yes", "Yes", "Yes", "Yes", "Yes"]
}

df = pd.DataFrame(data)
display(df.head())

,Day,Outlook,Temperature,Humidity,Wind,Play
0,D1,Sunny,Mild,Normal,Weak,Yes
1,D2,Sunny,Mild,Normal,Weak,Yes
2,D3,Sunny,Mild,Normal,Weak,Yes
3,D4,Sunny,Cool,High,Weak,Yes
4,D5,Sunny,Mild,High,Weak,Yes


In [5]:
def learn_concept_candidate_elimination(df):
    # Initialize S (specific hypothesis) and G (general hypothesis)
    # S starts with the first positive example
    # G starts with the most general hypothesis (all '?'s)

    # Assuming the last column is the target concept, and the rest are attributes
    attributes = df.columns[:-1]
    target = df.columns[-1]

    # Get positive examples (where target is 'Yes')
    positive_examples = df[df[target] == 'Yes'].iloc[:, :-1].values

    if len(positive_examples) == 0:
        return [], [] # No positive examples to learn from

    # Initialize S with the first positive example
    S = [list(positive_examples[0])]

    # Initialize G with the most general hypothesis
    G = [['?' for _ in attributes]]

    print("\nInitialization:")
    print(f"S0: {S}")
    print(f"G0: {G}")

    # Iterate through all positive examples
    for i, example in enumerate(positive_examples):
        print(f"\nProcessing positive example {i+1}: {list(example)}")

        # Process positive example for S
        S_new = []
        for s in S:
            # If s is not consistent with example, generalize it
            if not is_consistent(s, example):
                generalizations = generalize_hypothesis(s, example)
                S_new.extend(generalizations)
            else:
                S_new.append(s)
        S = remove_more_general(S_new)

        # Process positive example for G
        G_new = []
        for g in G:
            # If g is consistent with example, keep it
            if is_consistent(g, example):
                G_new.append(g)
            # If g is not consistent with example, it means a negative example (not in this dataset) would have pruned it
            # For purely positive examples, G should shrink to match the specific example
            # However, with only positive examples, G typically stays very general unless a specific negative example forces specialization.
            # For this dataset where all are 'Yes', G will remain general.

        G = G_new if G_new else G # If G_new is empty, keep previous G (shouldn't happen with only positive examples)
        G = remove_more_specific(G)

        print(f"S{i+1}: {S}")
        print(f"G{i+1}: {G}")

    return S, G

def is_consistent(hypothesis, example):
    for i, h_val in enumerate(hypothesis):
        if h_val != '?' and h_val != example[i]:
            return False
    return True

def generalize_hypothesis(specific_h, example):
    # Generalize specific_h to cover example
    new_h = list(specific_h)
    for i, h_val in enumerate(specific_h):
        if h_val != example[i]:
            new_h[i] = '?'
    return [new_h]

def remove_more_general(hypotheses):
    # Remove redundant hypotheses (those more general than others)
    filtered_hypotheses = []
    for h1 in hypotheses:
        is_redundant = False
        for h2 in hypotheses:
            if h1 != h2 and is_more_general(h2, h1):
                is_redundant = True
                break
        if not is_redundant:
            filtered_hypotheses.append(h1)
    return filtered_hypotheses

def remove_more_specific(hypotheses):
    # Remove redundant hypotheses (those more specific than others)
    filtered_hypotheses = []
    for h1 in hypotheses:
        is_redundant = False
        for h2 in hypotheses:
            if h1 != h2 and is_more_general(h1, h2):
                is_redundant = True
                break
        if not is_redundant:
            filtered_hypotheses.append(h1)
    return filtered_hypotheses

def is_more_general(h1, h2):
    # Returns True if h1 is more general than h2
    # A hypothesis h1 is more general than h2 if h1 covers all examples covered by h2,
    # and h1 covers at least one example not covered by h2.
    # In terms of attribute values, h1 is more general if for every attribute,
    # h1's value is '?' or matches h2's value, and there's at least one '?' in h1 where h2 has a specific value.

    if h1 == h2: # same hypothesis, not strictly more general
        return False

    h1_is_general = True
    h1_strictly_general = False

    for i in range(len(h1)):
        if h1[i] == '?':
            if h2[i] != '?': # h1 has '?' where h2 has a specific value
                h1_strictly_general = True
        elif h2[i] == '?': # h1 has a specific value where h2 has '?' (h1 is more specific)
            h1_is_general = False
            break
        elif h1[i] != h2[i]: # Mismatch in specific values
            h1_is_general = False
            break

    return h1_is_general and h1_strictly_general


In [6]:
S_final, G_final = learn_concept_candidate_elimination(df)

print("\nFinal Specific Hypothesis (S):")
for s in S_final:
    print(s)

print("\nFinal General Hypothesis (G):")
for g in G_final:
    print(g)


Initialization:
S0: [['D1', 'Sunny', 'Mild', 'Normal', 'Weak']]
G0: [['?', '?', '?', '?', '?']]

Processing positive example 1: ['D1', 'Sunny', 'Mild', 'Normal', 'Weak']
S1: [['D1', 'Sunny', 'Mild', 'Normal', 'Weak']]
G1: [['?', '?', '?', '?', '?']]

Processing positive example 2: ['D2', 'Sunny', 'Mild', 'Normal', 'Weak']
S2: [['?', 'Sunny', 'Mild', 'Normal', 'Weak']]
G2: [['?', '?', '?', '?', '?']]

Processing positive example 3: ['D3', 'Sunny', 'Mild', 'Normal', 'Weak']
S3: [['?', 'Sunny', 'Mild', 'Normal', 'Weak']]
G3: [['?', '?', '?', '?', '?']]

Processing positive example 4: ['D4', 'Sunny', 'Cool', 'High', 'Weak']
S4: [['?', 'Sunny', '?', '?', 'Weak']]
G4: [['?', '?', '?', '?', '?']]

Processing positive example 5: ['D5', 'Sunny', 'Mild', 'High', 'Weak']
S5: [['?', 'Sunny', '?', '?', 'Weak']]
G5: [['?', '?', '?', '?', '?']]

Final Specific Hypothesis (S):
['?', 'Sunny', '?', '?', 'Weak']

Final General Hypothesis (G):
['?', '?', '?', '?', '?']
